# Question

In the Premier League, how strongly asosciated is a team's annual wages associated with their performance in a season? Additionally, do clubs beat or miss what their wages predicted?

INTEREST: 

A club with a lot to spend might still be outperformed by smarter teams in recruitment, coaching, and overall well-being of players. I think back to Leicester City's run back in 2015-2016, where the underdogs had one of the smaller annual wages, but the coach's tactics and team's performance proved otherwise. 

WAGES:

I collected the data from FBref on October 16, 2026. Annual wages are defined in pounds and are estimated for earlier years because of the unkown circumstances of the clubs' wages. The scope is 10 seasons (2013-2024), where there are 200 club-seasons. the numbers are not an esitmate of first-team pay and are not audited club payroll 

# PREDICTIONS:

I expect teams like Manchecter City, Manchester United, Chelsea, Liverpool, Tottenham, and Arsenal to overperform their wages based on the clubs' reputations and teams that are predicted to perform poorly based on performance stats will most likely underperform their wages. However, I expect the relationship to be relatively strong because the more time owners put into making payment to keep players around should improve the team's overall performance. There are some confounding variables to be aware of, like the relationship between owner or club, or the players having beef between each other. 

In [62]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px


from season_loader import load_season
from validators import validate_season

In [59]:
df = pd.read_csv(Path('data/processed/merged_datasets/merged_wages_points.csv'))

In [60]:
print(df.head())
print(df.shape)
print(df.dtypes)
print(df.columns)


   Wage_Rk            Squad  # Pl  % Estimated   Season  Points_Rk  MP   W  D  \
0        1  Manchester City    26          100  2013_14          1  38  27  5   
1        2          Chelsea    30          100  2013_14          3  38  25  7   
2        3   Manchester Utd    31          100  2013_14          7  38  19  7   
3        4          Arsenal    32          100  2013_14          4  38  24  7   
4        5        Liverpool    26          100  2013_14          2  38  26  6   

    L  GD  Pts  Wages_GBP  Wages_EUR  Wages_USD  
0   6  65   86  113254000  135062723  137628660  
1   6  44   82  102922000  122741146  125072997  
2  12  21   64   97292000  116027007  118231300  
3   7  27   79   71994000   85857503   87488634  
4   6  51   84   65647000   78288296   79775627  
(200, 15)
Wage_Rk         int64
Squad          object
# Pl            int64
% Estimated     int64
Season         object
Points_Rk       int64
MP              int64
W               int64
D               int64
L    

In [61]:
print(df['Season'].nunique())    # Checks the number of unique seasons in the dataset
print(df['Season'].value_counts()) # Checks the count of each unique season in the dataset
print(df['Wages_GBP'].describe())  # Provides descriptive statistics for the 'Wages_GBP' column

10
Season
2013_14    20
2014_15    20
2015_16    20
2016_17    20
2017_18    20
2018_19    20
2019_20    20
2020_21    20
2021_22    20
2022_23    20
Name: count, dtype: int64
count    2.000000e+02
mean     7.252417e+07
std      4.382432e+07
min      1.558100e+07
25%      4.127500e+07
50%      5.593750e+07
75%      9.474700e+07
max      2.388500e+08
Name: Wages_GBP, dtype: float64


# DERIVED VARIABLES
Raw wages aren't comparable across years because of inflation over the years, and the distribution would be heavily skewed_right. 

In [55]:
df['Wage_Share'] = df.groupby('Season')['Wages_GBP'].transform(lambda x: x / x.sum())
print(df.groupby('Season')['Wage_Share'].sum())  # This should print 1.0 for each season if the calculation is correct
# Wage share helps handle inflation between seasons by normalizing wages within each season. It allows for a fair comparison of wages across different seasons, as it accounts for the total wage pool in each season rather than absolute values.

Season
2013_14    1.0
2014_15    1.0
2015_16    1.0
2016_17    1.0
2017_18    1.0
2018_19    1.0
2019_20    1.0
2020_21    1.0
2021_22    1.0
2022_23    1.0
Name: Wage_Share, dtype: float64


In [56]:
df['Log_Wage_Share'] = df['Wage_Share'].apply(lambda x: np.log(x) if x > 0 else np.nan)  # Apply log transformation to 'Wage_Share', handling non-positive values
print(df['Log_Wage_Share'].describe())  # Provides descriptive statistics for the 'Log_Wage_Share' column

count    200.000000
mean      -3.140276
std        0.537023
min       -4.418562
25%       -3.512740
50%       -3.176418
75%       -2.728854
max       -1.964533
Name: Log_Wage_Share, dtype: float64


In [57]:
df['Wages_Rank'] = df.groupby('Season')['Wages_GBP'].rank(method='min')

# Exploratory Data Analysis

## Distributions:

In [8]:
# Histogram of Raw Wages
fig1 = px.histogram(df, x='Wages_GBP', title='Distribution of Raw Wages', labels={'Wages_GBP': 'Wages (GBP)', 'y': 'Number of Teams'})
fig1.show()


Observations: The distribution is skewed right with more teams' annual wages ranging around the 50 million pounds. There are two teams as outliers that go beyond 200 million pounds, but the average annual wages is greater than the median of the wages given amongst clubs.  

In [64]:
# Histogram of Log Wage Share
df['Wage_Share'] = df.groupby('Season')['Wages_GBP'].transform(lambda x: x / x.sum())
df['Log_Wage_Share'] = df['Wage_Share'].apply(lambda x: np.log(x) if x > 0 else np.nan)
fig2 = px.histogram(df, x ='Log_Wage_Share', title='Distribution of Log Wage Share', labels={'Log_Wage_Share': 'Log Wage Share', 'y': 'Number of Teams'})
fig2.show()

Observations: Logging the values of the Wage Shares creates a more normal distribution, making the ratios more equidistant. 

In [66]:
# Histogram of Points
fig3 = px.histogram(df, x='Pts', title='Distribution of Points Earned', labels={'Pts': 'Points', 'y': 'Number of Teams'})
fig3.show()

Observations: The points are little bit skewed to the right where most teams appear to be performing below standards compared to their wages. The median number of points would be around 40-50 points which is a little surprising because a lot of owners invest a lot at high levels to make sure teams have access to perform stronger. 

## League-wide wages over time

In [67]:
# Line Plot of Average Wages Per Season
avg_wages_per_season = df.groupby('Season')['Wages_GBP'].mean().reset_index()
fig4 = px.line(avg_wages_per_season, x='Season', y='Wages_GBP', title='Average Wages Per Season', labels={'Wages_GBP': 'Average Wages (GBP)', 'Season': 'Season'})
fig4.show()

Observations: The massive jump of less than 50 million pound average wages compared to the above-90 million pound averages suggests the need to create Wage Share data to account for inflation. 

## Scatter Bubble Plot

In [82]:
# Scatterplot of Raw Wages vs Points (Clubs are hovered over to show club names)
fig5 = px.scatter(df, x='Wages_GBP', y='Pts', hover_data=['Squad', 'Season'], color = 'Season', title='Raw Wages vs Points', labels={'Wages_GBP': 'Wages (GBP)', 'Pts': 'Points'})
fig5.show()

**Observations:** There seems to be a correlation between teams spending more on wages anually performing better, where teams in-between 100 to 150 million earn over 80 points which is enough for Champions League Football. However, some teams that break the budget end up underperforming, causing pressure of big spending equaling poor results. 

In [83]:
# Scatterplot of Wage Share vs Points (Clubs are hovered over to show club names)
fig6 = px.scatter(df, x='Wage_Share', y='Pts', hover_data=['Squad', 'Season'], color = 'Season', title='Wage Shares vs Points', labels={'Share of league wage bill': 'Share of league wage bill', 'Pts': 'Points'})
fig6.show()

**Observations:** The x-axis shows that club’s share of the league’s total wage bill in that season, written as a fraction instead of a percentage. Most dots sit between about 0.02 and 0.06. Only a few stretch out toward 0.10 to 0.14, with the farthest at about 14% (probably Manchester United in 2021-22, given the 58 points, but hover to confirm). That long right tail is the skew the log transform is meant to address. Because this axis is linear, the crowded left side is hard to read and a few rich clubs stretch the right side out.

In [84]:
# Scatterplot of Log Wage Share vs Points (Clubs are hovered over to show club names)
fig7 = px.scatter(df, x='Log_Wage_Share', y='Pts', hover_data=['Squad', 'Season'], color = 'Season', title='Log Wage Shares vs Points', labels={'Log_Wage_Share': 'Log Wage Share', 'Pts': 'Points'})
fig7.show()

**Oberservations:** As the log accounts for inflation, low spenders are limited in how many points they get, while big spenders can finish anywhere from title winners to underachievers. For Chelsea (2022-2023), they spent high compared to other teams' annual wages, but the big spending caused them to fall short from even reaching European competitions.

## Correlation

From now on, I'll use the Low Wage Shares since it's the closest dataset that relates to straight line.

In [85]:
# CORRELATION ANALYSIS with Spearman's rank correlation coefficient and Pearson's correlation coefficient
# Spearman's rank correlation coefficient
spearman_corr = df[['Log_Wage_Share', 'Pts']].corr(method='spearman')
# Pearson's correlation coefficient
pearson_corr = df[['Log_Wage_Share', 'Pts']].corr(method='pearson')
print("Spearman's rank correlation coefficient:\n", spearman_corr)
print("Pearson's correlation coefficient:\n", pearson_corr)

Spearman's rank correlation coefficient:
                 Log_Wage_Share       Pts
Log_Wage_Share        1.000000  0.713033
Pts                   0.713033  1.000000
Pearson's correlation coefficient:
                 Log_Wage_Share       Pts
Log_Wage_Share        1.000000  0.715853
Pts                   0.715853  1.000000


In [86]:
# Small Table comparing Pearson's and Spearman's correlation coefficients
correlation_comparison = pd.DataFrame({
    'Correlation Method': ['Spearman', 'Pearson'],
    'Correlation Coefficient': [spearman_corr.loc['Log_Wage_Share', 'Pts'], pearson_corr.loc['Log_Wage_Share', 'Pts']]
})
print(correlation_comparison)

  Correlation Method  Correlation Coefficient
0           Spearman                 0.713033
1            Pearson                 0.715853


**Observations:** Both Spearman's and Pearson's Correlations resulted in similar coeffcients, where around 0.71 is good enough to say that some high-spending teams perform better overall. However, correlation isn’t causation. Higher-paid teams may score more because their coaches' tactices, hold certain positions, or are just historically better teams, and wages are often set before the points are earned. Spearman is noticeably higher than Pearson since it doesn't account for outliers. 

In [87]:
# Per Season Correlation Analysis
# Group the data by season and calculate the correlation for each season
seasonal_correlation = df.groupby('Season')[['Log_Wage_Share', 'Pts']].corr(method='spearman').iloc[0::2, -1].reset_index()
print(seasonal_correlation)

    Season         level_1       Pts
0  2013_14  Log_Wage_Share  0.731854
1  2014_15  Log_Wage_Share  0.659142
2  2015_16  Log_Wage_Share  0.487768
3  2016_17  Log_Wage_Share  0.713372
4  2017_18  Log_Wage_Share  0.872976
5  2018_19  Log_Wage_Share  0.810384
6  2019_20  Log_Wage_Share  0.662651
7  2020_21  Log_Wage_Share  0.808127
8  2021_22  Log_Wage_Share  0.815344
9  2022_23  Log_Wage_Share  0.533835


In [88]:
# Bar Graph of Correlation Coefficients Per Season
fig8 = px.bar(seasonal_correlation, x='Season', y='Pts', title='Spearman Correlation Coefficient of Log Wage Share vs Points Per Season', labels={'Pts': 'Spearman Correlation Coefficient', 'Season': 'Season'})
fig8.show()

**Oberservations:** Every season is clearly positive, and most sit between about 0.65 and 0.85. Clubs that spend a bigger share of the league’s wages reliably finish higher up the table. That fits what we know about the league: a handful of financially dominant clubs, and a wage bill that mostly reflects a club’s ability to attract better players. 2017-18 (about 0.87) is the peak. The table was close to a wage-ranked ladder, with the big spenders at the top. Manchester City’s record-breaking 100-point season came with one of the league’s largest wage bills. Lower coefficients suggest seasons where wages were a worse guide to results, which could mean more upsets, more inefficient spending, or a few clubs performing well outside their means. There’s no clear upward or downward trend over the decade, so this chart doesn’t support a simple story that money has become more or less decisive over time.

In [97]:
# Wage Rank v Points Rank Scatterplot with a diagonal line meaning “finished exactly where wages predicted.”
fig9 = px.scatter(df, x='Wage_Rk', y='Points_Rk', title='Wage Rank vs Points Rank', hover_data=['Squad', 'Season'], labels={'Wage_Rk': 'Wage Rank', 'Points_Rk': 'Points Rank'}, color='Season')
fig9.add_shape(type='line', x0=1, y0=1, x1=len(df), y1=len(df), line=dict(color='red', width=2))
fig9.show()


**Observations:** Clubs with wage ranks 1 to 5 almost always finish in the top eight, so the big spenders are fairly reliable. Spread widens sharply from about rank 8 downward, meaning wage rank says much less about where a mid or low spender will finish. A top-wage club can’t finish better than 1st, and a bottom-wage club can’t finish worse than 20th, so deviations near the edges are forced in one direction.

## Outliers

In [101]:
# Sort club-seasons by the gap between wage rank and league position, and display the top 10 overperformers and underperformers
df['Rank_Gap'] = df['Wage_Rk'] - df['Points_Rk']
underperformers = df.nsmallest(10, 'Rank_Gap')
overperformers = df.nlargest(10, 'Rank_Gap')
print("Top 10 Overperformers:\n", overperformers[['Squad', 'Season', 'Wage_Rk', 'Points_Rk', 'Rank_Gap']])
print("\nTop 10 Underperformers:\n", underperformers[['Squad', 'Season', 'Wage_Rk', 'Points_Rk', 'Rank_Gap']])
# Graphical representation of the top 10 overperformers and underperformers
fig10 = px.bar(overperformers, x='Squad', y='Rank_Gap', title='Top 10 Overperformers (Wage Rank vs Points Rank)', labels={'Rank_Gap': 'Rank Gap (Wage Rank - Points Rank)', 'Squad': 'Club'}, hover_data=['Season'])
fig10.show()
fig11 = px.bar(underperformers, x='Squad', y='Rank_Gap', title='Top 10 Underperformers (Wage Rank vs Points Rank)', labels={'Rank_Gap': 'Rank Gap (Wage Rank - Points Rank)', 'Squad': 'Club'}, hover_data=['Season'])
fig11.show()

Top 10 Overperformers:
                 Squad   Season  Wage_Rk  Points_Rk  Rank_Gap
56     Leicester City  2015_16       17          1        16
198          Brighton  2022_23       19          6        13
38       Swansea City  2014_15       19          8        11
117            Wolves  2018_19       18          7        11
139  Sheffield United  2019_20       20          9        11
199         Brentford  2022_23       20          9        11
78        Bournemouth  2016_17       19          9        10
158      Leeds United  2020_21       19          9        10
19     Crystal Palace  2013_14       20         11         9
37     Crystal Palace  2014_15       18         10         8

Top 10 Underperformers:
               Squad   Season  Wage_Rk  Points_Rk  Rank_Gap
28              QPR  2014_15        9         20       -11
166         Everton  2021_22        7         16        -9
182         Chelsea  2022_23        3         12        -9
187         Everton  2022_23        8      

**Overperformers:** Teams like Leicester City and Brighton have massive gaps between wage rankings and points rankings, indicating a more focused approach on coaching and player's individual performances. Sports analysis from various seasons show that overperforming teams focusing more on teamwork and tikki-takka football. 

**Underperformer:** These teams spent big on keeping players around. However, many of the new players had to form new chemistries with new teammates, and the lack of coordination made them vulnerable to well-established teams. 

## Club-Level Patterns

In [103]:
# Average wage share and average points per club across the 10 seasons
avg_wage_share_points = df.groupby('Squad')[['Wage_Share', 'Pts']].mean().reset_index()
avg_wage_share_points = avg_wage_share_points.sort_values(by='Wage_Share', ascending=False)
print(avg_wage_share_points)
# Graphical representation of average wage share and average points per club across the 10 seasons
fig12 = px.scatter(avg_wage_share_points, x='Wage_Share', y='Pts', title='Average Wage Share vs Average Points Per Club (Across 10 Seasons)', labels={'Wage_Share': 'Average Wage Share', 'Pts': 'Average Points'}, hover_data=['Squad'])
fig12.show()

               Squad  Wage_Share        Pts
17    Manchester Utd    0.113979  67.700000
16   Manchester City    0.100338  83.400000
7            Chelsea    0.090836  72.800000
0            Arsenal    0.079482  71.500000
15         Liverpool    0.075725  78.200000
28         Tottenham    0.059436  69.800000
9            Everton    0.051906  52.500000
31          West Ham    0.045047  49.100000
14    Leicester City    0.042280  52.888889
8     Crystal Palace    0.041935  44.600000
22               QPR    0.040328  30.000000
26        Sunderland    0.038109  34.750000
1        Aston Villa    0.037412  41.285714
19         Newcastle    0.036716  46.444444
24       Southampton    0.036320  47.000000
25        Stoke City    0.034913  48.600000
30         West Brom    0.032396  39.833333
10            Fulham    0.031773  34.500000
32            Wolves    0.030143  50.600000
29           Watford    0.029759  38.666667
18     Middlesbrough    0.028958  28.000000
27      Swansea City    0.028898

**Oberservations:** There’s a clear gap. Six clubs sit at the top right, with average wage shares of roughly 0.06 to 0.115 and average points of about 68 to 83. Everyone else is bunched below about 0.05. This looks like the familiar “big six” structure: a small group with much larger wage bills that consistently earns far more points than the rest. Because of their history, the "Big Six" (Chelsea, Man Utd, Man City, Tottenham, Arsenal, Liverpool) tend to keep their reputations strong with new players interested with sigining. Across the whole cloud the trend is clearly positive, but within the dense cluster (wage share about 0.015 to 0.05) it’s noisy and fairly flat. Clubs there range from about 28 to 53 points with little visible order. So wages separate the elite from the rest much better than they rank mid-table and lower clubs against each other. This matches the earlier rank plot, where the spread widened below the top five or so. The highest wage share (about 0.115) corresponds to only about 67 points, well below clubs spending less. Two things could explain it: there’s a ceiling on points, and a large wage bill can reflect inefficient or accumulated spending rather than current quality. 

In [104]:
# Check number of players relates to wages or points
player_count = df.groupby('Squad')['# Pl'].first()
wage_share = df.groupby('Squad')['Wage_Share'].mean()
points = df.groupby('Squad')['Pts'].mean()

# Create a DataFrame with the aggregated data
aggregated_data = pd.DataFrame({
    '# Pl': player_count,
    'Wage_Share': wage_share,
    'Points': points
})

print(aggregated_data)


                  # Pl  Wage_Share     Points
Squad                                        
Arsenal             32    0.079482  71.500000
Aston Villa         29    0.037412  41.285714
Bournemouth         39    0.026349  42.000000
Brentford           90    0.014886  52.500000
Brighton            24    0.027470  45.000000
Burnley             27    0.022146  40.142857
Cardiff City        30    0.024786  32.000000
Chelsea             30    0.090836  72.800000
Crystal Palace      34    0.041935  44.600000
Everton             32    0.051906  52.500000
Fulham              35    0.031773  34.500000
Huddersfield        25    0.019385  16.000000
Hull City           26    0.028728  35.333333
Leeds United        85    0.018391  42.666667
Leicester City      34    0.042280  52.888889
Liverpool           26    0.075725  78.200000
Manchester City     26    0.100338  83.400000
Manchester Utd      31    0.113979  67.700000
Middlesbrough       25    0.028958  28.000000
Newcastle           26    0.036716

**Observations:** It doesn't seem like number of players relates to a club's wage share or points because some higher-ranked teams outperformed based on a smaller squad size. Most of the points accumulated from team performance rather than size of team. 

## Wrap-Up:

The relationship looks relatively strong, especially through the Spearman and Pearson coefficient tests because most teams with big wages tend to outperform smaller-budget teams. The higher wages usually come from prestige teams, where rising-star are attracted to the club's history, thus allowing the club to spend more for a player. However, a few outliers highlighted how teams with stronger morals, better chemistry, and advanced performing can outperform how much they spent, like Leiceister City's miracle. However, even after that run, higher-budget teams were quick to invest in those star players, demoting the potential for smaller clubs to really succeed. The scale is roughly linear with other outliers of high-budget overdoing it with big spendings, but lacked team chemistry to actually play the sport.   